# VeriAudit v2: exploratory v0.1 replication (Kaggle, free 2 × T4)

Runs the four v2 confirmatory models on the 30 VeriAudit v0.1 harmful requests in their four human-written forms (120 prompts × greedy + 3 samples), then two judges. **Exploratory, not the main study.** Free tier; no paid compute.

**Before Run All (one time, ~10 min):**
1. Kaggle account with **phone verification** (needed for GPUs).
2. Notebook settings (right panel): **Accelerator = GPU T4 ×2**, **Internet = on**.
3. Hugging Face: accept the licence on https://huggingface.co/CohereLabs/aya-expanse-8b, create a **read** token (Settings → Access Tokens).
4. Kaggle: **Add-ons → Secrets → Add** `HF_TOKEN` = your token, and tick it for this notebook.

Then **Run All**. Expect roughly 2–4 hours. When it finishes, download `veriaudit_exploratory_v01.zip` from the Output panel and attach it in the chat.

In [ ]:
import os
import subprocess

from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_HOME"] = "/kaggle/tmp/hf"  # large scratch disk; cleared after each model
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["PYTHONPATH"] = "backend"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /kaggle/working
!rm -rf veriaudit && git clone -q -b veriaudit-x-dataset https://github.com/abeeranajam31/veriaudit.git
%cd veriaudit
!git log --oneline -1
import sys
print(sys.version)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'backend/requirements-v2.txt', 'vllm', 'huggingface_hub'], capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-4000:])
if r.returncode:
    raise RuntimeError('pip install failed - see the error above')
r = subprocess.run([sys.executable, '-c', 'import vllm, torch; print("vllm", vllm.__version__, "torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.device_count())'], capture_output=True, text=True)
print(r.stdout, r.stderr[-3000:])
if r.returncode:
    raise RuntimeError('vllm/torch import failed - see the error above')

In [ ]:
# access check: every pinned repo must be reachable with this token (fails fast on a missing licence)
import yaml
from huggingface_hub import HfApi

api = HfApi(token=os.environ["HF_TOKEN"])
specs = [m for m in yaml.safe_load(open("configs/models_kaggle_t4.yaml"))["models"] if m["backend"] == "vllm"]
specs += yaml.safe_load(open("configs/judges_kaggle_t4.yaml"))["judges"]
for s in specs:
    api.model_info(s["model_id"], revision=s["revision"])
    api.auth_check(s["model_id"])
    print("ok ", s["model_id"], s["revision"][:10])

In [ ]:
# 1. generation: one model per process (frees GPU memory), cache deleted after each model.
# A model that fails on T4 is reported and skipped; the run can be resumed by re-running this cell.
import shutil

CFG = "configs/experiments/exploratory_v01_panel.yaml"
failed = []
for key in ["qwen3_8b", "aya_expanse_8b", "qwen3_5_9b", "gemma4_e4b"]:
    print(f"=== {key}", flush=True)
    r = subprocess.run(["python", "-m", "app.v2.inference.runner", CFG, "--models", key], capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-3000:] if r.returncode else "")
    shutil.rmtree("/kaggle/tmp/hf/hub", ignore_errors=True)
    if r.returncode:
        failed.append(key)
if len(failed) == 4:
    raise RuntimeError("every model failed - stopping before the judges; see the errors above")
print("failed models (skipped):", failed or "none")

In [ ]:
# record one manifest covering all four models (instant: completed generations are skipped)
!python -m app.v2.inference.runner configs/experiments/exploratory_v01_panel.yaml --models qwen3_8b aya_expanse_8b qwen3_5_9b gemma4_e4b 2>&1 | tail -3

In [ ]:
# 2. judges (triage labels; checked against a human sample afterwards)
RUN = "results/v2/runs/exploratory-v01-panel"
for judge in ["mistral_nemo_12b", "granite_3_3_8b"]:
    print(f"=== {judge}", flush=True)
    r = subprocess.run(
        ["python", "-m", "app.v2.judges.run_judge", "--run", RUN, "--judge", judge, "--judges", "configs/judges_kaggle_t4.yaml"], capture_output=True, text=True
    )
    print(r.stdout[-1500:], r.stderr[-3000:] if r.returncode else "")
    shutil.rmtree("/kaggle/tmp/hf/hub", ignore_errors=True)

In [ ]:
# 3. package everything for download (attach this zip in the chat)
import json

m = json.load(open(f"{RUN}/run_manifest.json"))
print(m["status"], m.get("n_succeeded"), "/", m.get("n_planned"), "failed:", m.get("n_failed_after_retries"))
!cd /kaggle/working/veriaudit && zip -qr /kaggle/working/veriaudit_exploratory_v01.zip results/v2/runs/exploratory-v01-panel results/v2/labels
!ls -lh /kaggle/working/veriaudit_exploratory_v01.zip